In [ ]:
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, learning_curve, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from xgboost import XGBRegressor
import joblib

# Load and prepare data
dataset_path = "PATH_TO_YOUR_DATASET/cleaned_data.csv"  # Update with your dataset path
df = pd.read_csv(dataset_path, encoding='ISO-8859-1')
df = df.drop(columns=['Food Name'])
data = df.dropna()

# Separate features and target
X = data.drop('Glycemic Index', axis=1)
y = data['Glycemic Index']

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Scale features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
def plot_learning_curve(model, model_name, X, y):
    """
    Plot and save learning curves for model performance visualization.
    
    Args:
        model: Sklearn model object
        model_name (str): Name of the model for plot title
        X: Feature matrix
        y: Target variable
    """
    # Calculate learning curve scores
    train_sizes, train_scores, valid_scores = learning_curve(
        model, X, y, 
        cv=5, 
        scoring='neg_mean_squared_error', 
        train_sizes=np.linspace(0.1, 0.9, 10)
    )

    # Convert negative MSE to positive values for readability
    train_scores_mean = -np.mean(train_scores, axis=1)
    valid_scores_mean = -np.mean(valid_scores, axis=1)

    # Plot Learning Curve
    plt.figure(figsize=(8, 5))
    plt.plot(train_sizes, train_scores_mean, 'o-', label="Training Error")
    plt.plot(train_sizes, valid_scores_mean, 'o-', label="Validation Error")
    plt.xlabel("Training Size")
    plt.ylabel("Mean Squared Error")
    plt.title(f"{model_name} - Learning Curve")
    plt.legend()
    
    # Save and show plot
    lc_filename = f"{model_name.lower().replace(' ', '_')}_learning_curve.png"
    plt.savefig(lc_filename, bbox_inches='tight')
    print(f"Saved: {lc_filename}")
    plt.show()
    plt.close()

In [ ]:
def evaluate_model(model, model_name):
    """
    Evaluate machine learning model with various metrics and visualizations.
    
    Args:
        model: Trained model object (XGBoost or sklearn model)
        model_name (str): Name of the model for plots and saving files
        
    Outputs:
        - Performance metrics
        - Learning curve plot
        - Actual vs Predicted plot
        - Residual distribution plot
        - Feature importance plot (if available)
        - Saved model file
    """
    # Model Training
    if isinstance(model, XGBRegressor):
        model.fit(
            X_train_scaled, 
            y_train,
            eval_set=[(X_test_scaled, y_test)], 
            early_stopping_rounds=10,
            verbose=False
        )
    else:
        model.fit(X_train_scaled, y_train)
    
    # Generate Predictions
    y_pred = model.predict(X_test_scaled)
    y_train_pred = model.predict(X_train_scaled)
    
    # Calculate Performance Metrics
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    cv_scores = cross_val_score(model, X_train_scaled, y_train, cv=5, scoring='r2')
    
    # Print Performance Metrics
    print(f"\n{model_name} Metrics:")
    print(f"Mean Squared Error: {mse:.4f}")
    print(f"Root Mean Squared Error: {rmse:.4f}")
    print(f"Mean Absolute Error: {mae:.4f}")
    print(f"R^2: {r2:.4f}")
    print(f"Cross-Validation R^2 (mean): {cv_scores.mean():.4f}")
    print(f"Cross-Validation R^2 (std): {cv_scores.std():.4f}")
    
    # Generate Visualization Plots
    # 1. Learning Curve
    plot_learning_curve(model, model_name, X_train_scaled, y_train)
    
    # 2. Actual vs Predicted Plot
    plt.figure(figsize=(6, 4))
    plt.scatter(y_test, y_pred, alpha=0.5)
    plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2)
    plt.xlabel("Actual Values")
    plt.ylabel("Predicted Values")
    plt.title(f"{model_name} - Actual vs Predicted")
    actual_vs_pred_filename = f"{model_name.lower().replace(' ', '_')}_actual_vs_predicted.png"
    plt.savefig(actual_vs_pred_filename, bbox_inches='tight')
    print(f"Saved: {actual_vs_pred_filename}")
    plt.show()
    plt.close()
    
    # 3. Residual Distribution Plot
    residuals = y_test - y_pred
    plt.figure(figsize=(6, 4))
    sns.histplot(residuals, kde=True, bins=50)
    plt.xlabel("Residuals")
    plt.ylabel("Frequency")
    plt.title(f"{model_name} - Residual Distribution")
    lower_bound, upper_bound = np.percentile(residuals, [1, 99])
    plt.xlim(lower_bound, upper_bound)
    residuals_filename = f"{model_name.lower().replace(' ', '_')}_residual_distribution.png"
    plt.savefig(residuals_filename, bbox_inches='tight')
    print(f"Saved: {residuals_filename}")
    plt.show()
    plt.close()
    
    # 4. Feature Importance Plot (if available)
    if hasattr(model, "feature_importances_"):
        feat_imp = pd.DataFrame({
            'feature': X.columns,
            'importance': model.feature_importances_
        }).sort_values('importance', ascending=True)
        plt.figure(figsize=(6, 4))
        plt.barh(feat_imp['feature'], feat_imp['importance'])
        plt.title(f"{model_name} - Feature Importance")
        feat_imp_filename = f"{model_name.lower().replace(' ', '_')}_feature_importance.png"
        plt.savefig(feat_imp_filename, bbox_inches='tight')
        print(f"Saved: {feat_imp_filename}")
        plt.show()
        plt.close()
    
    # Save Model
    model_filename = f"{model_name.lower().replace(' ', '_')}.joblib"
    joblib.dump(model, model_filename)
    print(f"Model saved as: {model_filename}")

In [ ]:
# Initialize and evaluate Random Forest model
rf_model = RandomForestRegressor(
    n_estimators=100,    # Number of trees in the forest
    max_depth=None,      # Maximum depth of trees
    min_samples_split=2, # Minimum samples required to split internal node
    random_state=42      # Seed for reproducibility
)

# Train and evaluate model
evaluate_model(rf_model, "Random Forest")

In [ ]:
# Initialize and evaluate XGBoost model
xgb_model = XGBRegressor(
    n_estimators=100,    # Number of boosting rounds
    learning_rate=0.1,   # Learning rate
    max_depth=6,         # Maximum tree depth
    random_state=42,     # Seed for reproducibility
    early_stopping_rounds=10  # Stop if no improvement
)

# Train and evaluate model
evaluate_model(xgb_model, "XGBoost")

In [ ]:
# Save the StandardScaler object for future preprocessing
try:
    scaler_path = "PATH_TO_SAVE_DIR/scaler.joblib"  # Update with your save directory
    joblib.dump(scaler, scaler_path)
    print(f"Scaler saved successfully at: {scaler_path}")
except Exception as e:
    print(f"Error saving scaler: {str(e)}")